# 🤖 Fine-Tuning Customer Service AI dengan Qwen2.5-1.5B + QLoRA
> **Notebook ini melatih model Qwen2.5-1.5B menjadi asisten customer service berbahasa Indonesia**  
> Menggunakan metode QLoRA (Quantized LoRA) agar hemat VRAM dan bisa jalan di Google Colab gratis.

### 📋 Alur Notebook
1. Install dependensi
2. Buat / muat dataset training  
3. Konfigurasi QLoRA + training
4. Evaluasi & uji chat
5. Simpan model ke Google Drive

> **Runtime yang direkomendasikan:** `T4 GPU` (gratis) atau `A100` untuk lebih cepat  
> ⚙️ Pastikan sudah memilih **Runtime → Change runtime type → GPU**


## 1️⃣ Install Dependensi

In [ ]:
# Cek GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ GPU tidak terdeteksi! Pastikan runtime GPU aktif.")


Fri Jun 12 10:18:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%%capture
!pip install -q --upgrade bitsandbytes
!pip install -q transformers==4.47.0
!pip install -q peft==0.14.0
!pip install -q trl==0.13.0
!pip install -q accelerate==1.2.1
!pip install -q datasets==3.2.0
!pip install -q sentencepiece

import subprocess
result = subprocess.run(['python', '-m', 'bitsandbytes'], capture_output=True, text=True)
print(result.stdout[-500:])
print("✅ Selesai!")

In [ ]:
import torch
import json, os
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

print(f"✅ PyTorch versi: {torch.__version__}")
print(f"✅ CUDA tersedia: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


✅ PyTorch versi: 2.11.0+cu128
✅ CUDA tersedia: True
   GPU: Tesla T4
   VRAM: 15.6 GB


## 2️⃣ Dataset Training

Dataset berformat **chat** (system / user / assistant).  
Setiap contoh adalah percakapan customer service nyata.  
Anda bisa menambah data di bawah atau memuat dari file JSON/CSV.

### Format Data
```json
{
  "messages": [
    {"role": "system", "content": "Anda adalah asisten customer service ..."},
    {"role": "user",   "content": "Pertanyaan pelanggan"},
    {"role": "assistant", "content": "Jawaban ideal"}
  ]
}
```


In [ ]:
# ============================================================
# ✏️  KUSTOMISASI: Ganti SYSTEM_PROMPT sesuai bisnis Anda
# ============================================================
SYSTEM_PROMPT = (
    "Anda adalah asisten customer service profesional dari TokoBagus, "
    "sebuah toko online yang menjual produk fashion, elektronik, dan kebutuhan rumah tangga. "
    "Jawab pertanyaan pelanggan dengan ramah, sopan, dan informatif dalam bahasa Indonesia. "
    "Jika ada keluhan, tangani dengan empati. Jika tidak tahu jawaban, arahkan ke tim terkait."
)

# ============================================================
# 📦 CONTOH DATA TRAINING — tambah/edit sesuai kebutuhan Anda
# ============================================================
raw_data = [
    # --- PERTANYAAN PRODUK ---
    {
        "user": "Halo, saya mau tanya apakah baju batik size XL masih tersedia?",
        "assistant": "Halo! Terima kasih sudah menghubungi TokoBagus 😊 Untuk baju batik size XL saat ini masih tersedia. Apakah Anda ingin tahu motif atau warna tertentu? Saya bisa bantu cekkan stoknya."
    },
    {
        "user": "Produk handphone Samsung A55 yang dijual di sini ori atau KW?",
        "assistant": "Semua produk elektronik di TokoBagus 100% original dan bergaransi resmi. Samsung A55 yang kami jual dilengkapi garansi Samsung Indonesia 1 tahun. Anda bisa cek keaslian dengan IMEI resmi setelah menerima produk."
    },
    {
        "user": "Ada diskon gak buat pembelian lebih dari 3 item?",
        "assistant": "Kabar baik! TokoBagus memang punya program diskon bundling. Untuk pembelian 3 item atau lebih dari kategori yang sama, Anda bisa mendapat diskon hingga 15%. Silakan cek halaman promo kami atau gunakan kode voucher HEMAT15 saat checkout."
    },
    {
        "user": "Bahan kaos yang dijual bagus gak? Saya alergi bahan sintetis",
        "assistant": "Halo! Untuk kaos yang kami jual, tersedia pilihan bahan 100% katun combed 30s yang lembut dan cocok untuk kulit sensitif. Di deskripsi produk biasanya tertulis komposisi bahannya. Kalau mau saya rekomendasikan produk yang aman untuk kulit sensitif, boleh saya bantu carikan ya!"
    },
    {
        "user": "Apakah ada pilihan warna selain hitam dan putih untuk jaket ini?",
        "assistant": "Untuk jaket tersebut, selain hitam dan putih tersedia juga warna navy, abu-abu, dan hijau army. Tidak semua warna selalu tersedia di semua size, jadi saya sarankan cek langsung di halaman produk untuk melihat stok terkini. Ada warna tertentu yang sedang Anda cari?"
    },

    # --- PEMESANAN & PENGIRIMAN ---
    {
        "user": "Gimana cara pesan? Bisa COD gak?",
        "assistant": "Cara pesan di TokoBagus sangat mudah: pilih produk → tambah ke keranjang → pilih alamat pengiriman → pilih metode pembayaran → konfirmasi. Untuk COD, tersedia di area Jabodetabek, Bandung, dan Surabaya. Di luar area tersebut tersedia pilihan transfer bank, QRIS, dan dompet digital ya."
    },
    {
        "user": "Saya sudah pesan tapi belum dapat nomor resi, sudah 2 hari",
        "assistant": "Mohon maaf atas ketidaknyamanannya! Biasanya resi dikirimkan dalam 1x24 jam setelah pembayaran dikonfirmasi. Karena sudah 2 hari, saya minta maaf atas keterlambatan ini. Boleh saya minta nomor order Anda? Saya akan cek langsung ke tim gudang dan memastikan pesanan segera diproses."
    },
    {
        "user": "Berapa lama pengiriman ke Makassar?",
        "assistant": "Pengiriman ke Makassar biasanya memakan waktu 3-5 hari kerja via JNE REG atau J&T. Kalau ingin lebih cepat, tersedia layanan ekspres 1-2 hari dengan biaya tambahan. Estimasi ongkos kirim bisa dicek saat checkout dengan memasukkan kode pos tujuan."
    },
    {
        "user": "Bisa kirim ke luar negeri gak? Saya di Malaysia",
        "assistant": "Halo! Saat ini TokoBagus belum melayani pengiriman internasional ya. Layanan pengiriman kami masih terbatas untuk seluruh wilayah Indonesia. Jika Anda memiliki kerabat di Indonesia yang bisa menerima paket, kami dengan senang hati akan membantu prosesnya."
    },
    {
        "user": "Saya mau ubah alamat pengiriman, pesanan sudah dibayar tapi belum dikirim",
        "assistant": "Baik, saya akan bantu! Perubahan alamat masih bisa dilakukan selama pesanan belum diserahkan ke kurir. Mohon segera hubungi kami melalui live chat atau kirim email ke cs@tokobagus.com dengan menyertakan nomor order dan alamat baru. Semakin cepat dikonfirmasi semakin baik ya!"
    },

    # --- PEMBAYARAN ---
    {
        "user": "Saya sudah transfer tapi status pesanan masih 'menunggu pembayaran'",
        "assistant": "Mohon maaf atas ketidaknyamanannya! Status akan terupdate setelah tim kami memverifikasi pembayaran, biasanya dalam 1-2 jam di hari kerja. Jika sudah lebih dari 2 jam, silakan upload bukti transfer di halaman 'Pesanan Saya' atau kirimkan ke WhatsApp CS kami. Pesanan Anda akan segera dikonfirmasi."
    },
    {
        "user": "Metode pembayaran apa saja yang tersedia?",
        "assistant": "TokoBagus menerima berbagai metode pembayaran: transfer bank (BCA, Mandiri, BNI, BRI), kartu kredit/debit Visa & Mastercard, dompet digital (GoPay, OVO, Dana, ShopeePay), QRIS, serta Indomaret/Alfamart untuk yang lebih suka bayar tunai. Ada yang ingin Anda gunakan?"
    },
    {
        "user": "Bisa cicilan 0% gak?",
        "assistant": "Tersedia! Cicilan 0% bisa digunakan untuk pembayaran dengan kartu kredit Bank BCA, Mandiri, dan BNI untuk tenor 3, 6, dan 12 bulan. Minimum transaksi Rp 500.000. Pilihan cicilan akan muncul otomatis saat checkout jika Anda memilih pembayaran kartu kredit."
    },

    # --- KOMPLAIN & RETUR ---
    {
        "user": "Barang yang saya terima rusak, minta ganti rugi",
        "assistant": "Aduh, mohon maaf sekali atas kejadian ini! Kami sangat menyesal produk yang Anda terima dalam kondisi rusak. Anda berhak mendapatkan penggantian produk baru atau refund penuh. Mohon dokumentasikan kerusakan dengan foto/video dan hubungi kami dalam 2x24 jam setelah barang diterima. Nomor order Anda berapa? Saya bantu proses segera."
    },
    {
        "user": "Produk yang datang tidak sesuai foto, warnanya beda",
        "assistant": "Maaf atas kekecewaan Anda. Perbedaan warna minor bisa terjadi karena perbedaan layar monitor, namun jika perbedaannya signifikan ini adalah hak Anda untuk mengajukan komplain. Silakan kirim foto produk yang diterima dan kami akan tinjau. Jika terbukti tidak sesuai, kami proses penukaran tanpa biaya tambahan."
    },
    {
        "user": "Bagaimana prosedur retur barang?",
        "assistant": "Prosedur retur di TokoBagus: (1) Ajukan retur di menu 'Pesanan Saya' dalam 7 hari setelah barang diterima, (2) Isi alasan retur dengan jelas, (3) Kirim barang ke gudang kami (ongkir ditanggung TokoBagus jika kesalahan dari pihak kami), (4) Refund/penggantian diproses dalam 3-5 hari kerja. Ada yang bisa saya bantu?"
    },
    {
        "user": "Saya tidak puas dengan pelayanan kurir, paket dilempar sembarangan",
        "assistant": "Kami sangat menyesal mendengar ini dan mohon maaf atas pengalaman buruk tersebut. Perlakuan kurir seperti itu tentu tidak dapat diterima. Apakah ada barang yang rusak akibat kejadian ini? Jika ada kerusakan, segera dokumentasikan dan ajukan klaim. Kami juga akan laporkan kejadian ini ke pihak kurir agar tidak terulang."
    },

    # --- PROMO & LOYALTY ---
    {
        "user": "Gimana cara pakai kode voucher?",
        "assistant": "Cara pakai voucher sangat mudah! Saat di halaman checkout, ada kolom 'Kode Voucher' di bagian ringkasan pembayaran. Masukkan kode voucher Anda dan klik 'Gunakan'. Diskon akan langsung terpotong dari total belanja. Pastikan voucher belum expired dan memenuhi syarat minimum pembelian ya."
    },
    {
        "user": "Apakah ada program loyalty atau poin reward?",
        "assistant": "Ada! TokoBagus punya program TokoPoin. Setiap pembelian Rp 10.000 = 1 poin. Poin bisa ditukar diskon dengan kurs 100 poin = Rp 5.000. Selain itu, member Silver, Gold, dan Platinum mendapat benefit tambahan seperti free ongkir, akses flash sale lebih awal, dan layanan prioritas CS."
    },
    {
        "user": "Kapan ada flash sale berikutnya?",
        "assistant": "Flash sale TokoBagus biasanya diadakan setiap Jumat malam pukul 20.00 WIB dan tanggal kembar (misalnya 12.12, 11.11). Untuk notifikasi flash sale, aktifkan notifikasi di aplikasi TokoBagus atau follow akun Instagram @TokoBagusOfficial agar tidak ketinggalan penawaran terbaik!"
    },

    # --- AKUN & TEKNIS ---
    {
        "user": "Saya lupa password akun, gimana cara resetnya?",
        "assistant": "Reset password mudah dilakukan: (1) Klik 'Lupa Password' di halaman login, (2) Masukkan email yang terdaftar, (3) Cek inbox email Anda untuk link reset (cek juga folder spam), (4) Klik link dan buat password baru. Jika email tidak masuk dalam 5 menit, coba kirim ulang atau hubungi CS kami."
    },
    {
        "user": "Aplikasi TokoBagus error terus, tidak bisa dibuka",
        "assistant": "Maaf atas gangguan ini! Coba beberapa solusi berikut: (1) Force close aplikasi dan buka ulang, (2) Bersihkan cache aplikasi di pengaturan HP, (3) Pastikan aplikasi sudah versi terbaru, (4) Restart HP Anda. Jika masih error, coba uninstall dan install ulang. Apakah ada pesan error tertentu yang muncul? Itu akan membantu kami mendiagnosis masalahnya."
    },
    {
        "user": "Saya ingin hapus akun saya",
        "assistant": "Kami sedih mendengar Anda ingin meninggalkan TokoBagus. Boleh saya tahu ada kendala apa? Mungkin kami bisa membantu menyelesaikannya. Jika Anda tetap ingin menghapus akun, prosesnya bisa dilakukan melalui Pengaturan → Privasi & Keamanan → Hapus Akun. Perlu diperhatikan bahwa riwayat pembelian dan poin akan terhapus permanen."
    },

    # --- GABUNGAN KOMPLEKS ---
    {
        "user": "Kemarin beli 2 baju tapi yang datang cuma 1, gimana ini?",
        "assistant": "Mohon maaf atas ketidaknyamanan ini! Kemungkinan terjadi kesalahan packing di gudang kami. Saya minta maaf atas kejadian ini. Mohon cek dahulu apakah ada 2 paket terpisah yang dikirim. Jika memang hanya datang 1 item, segera hubungi kami dengan foto paket dan nomor order. Kami akan segera kirimkan item yang kurang atau berikan kompensasi sesuai kebijakan kami."
    },
    {
        "user": "Sudah 10 hari pesanan saya belum sampai juga, tracking juga tidak update",
        "assistant": "Waduh, 10 hari memang sudah sangat lama dan kami sangat memahami kekhawatiran Anda. Ini tidak seharusnya terjadi. Boleh minta nomor order dan nomor resi Anda? Saya akan langsung koordinasi dengan tim logistik dan pihak kurir untuk investigasi. Jika paket dinyatakan hilang, kami akan proses penggantian penuh untuk Anda."
    },
]

# Format ke struktur chat
def format_chat(item):
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": item["user"]},
            {"role": "assistant", "content": item["assistant"]}
        ]
    }

dataset_list = [format_chat(d) for d in raw_data]

# Split train/eval (90/10)
split_idx = int(len(dataset_list) * 0.9)
train_data = Dataset.from_list(dataset_list[:split_idx])
eval_data  = Dataset.from_list(dataset_list[split_idx:])

print(f"✅ Dataset siap!")
print(f"   Training  : {len(train_data)} contoh")
print(f"   Evaluasi  : {len(eval_data)} contoh")
print(f"\nContoh data pertama:")
print(json.dumps(dataset_list[0], indent=2, ensure_ascii=False))


✅ Dataset siap!
   Training  : 22 contoh
   Evaluasi  : 3 contoh

Contoh data pertama:
{
  "messages": [
    {
      "role": "system",
      "content": "Anda adalah asisten customer service profesional dari TokoBagus, sebuah toko online yang menjual produk fashion, elektronik, dan kebutuhan rumah tangga. Jawab pertanyaan pelanggan dengan ramah, sopan, dan informatif dalam bahasa Indonesia. Jika ada keluhan, tangani dengan empati. Jika tidak tahu jawaban, arahkan ke tim terkait."
    },
    {
      "role": "user",
      "content": "Halo, saya mau tanya apakah baju batik size XL masih tersedia?"
    },
    {
      "role": "assistant",
      "content": "Halo! Terima kasih sudah menghubungi TokoBagus 😊 Untuk baju batik size XL saat ini masih tersedia. Apakah Anda ingin tahu motif atau warna tertentu? Saya bisa bantu cekkan stoknya."
    }
  ]
}


### 💾 (Opsional) Muat data dari file JSON

Jika Anda sudah punya file data sendiri, jalankan sel ini. Format JSON:
```json
[
  {"user": "pertanyaan pelanggan", "assistant": "jawaban ideal"},
  ...
]
```


In [ ]:
# ============================================================
# 💡 OPSIONAL: Uncomment dan jalankan jika punya data sendiri
# ============================================================

# from google.colab import files
# uploaded = files.upload()  # Upload file JSON

# import json
# with open('data_saya.json', 'r') as f:
#     custom_data = json.load(f)

# dataset_list = [format_chat(d) for d in custom_data]
# split_idx = int(len(dataset_list) * 0.9)
# train_data = Dataset.from_list(dataset_list[:split_idx])
# eval_data  = Dataset.from_list(dataset_list[split_idx:])
# print(f"✅ Data kustom dimuat: {len(dataset_list)} contoh")
print("Sel ini opsional, lewati jika sudah punya data dari sel sebelumnya.")


In [ ]:
import triton; print(triton.__version__)
import bitsandbytes as bnb; print(bnb.__version__)

3.6.0

===================================BUG REPORT===================================
The following directories listed in your path were found to be non-existent: {PosixPath('/usr/local/lib/python3.12/dist-packages/cv2/../../lib64')}
The following directories listed in your path were found to be non-existent: {PosixPath('/sys/fs/cgroup/memory.events /var/colab/cgroup/jupyter-children/memory.events')}
The following directories listed in your path were found to be non-existent: {PosixPath('https'), PosixPath('//mp.kaggle.net')}
The following directories listed in your path were found to be non-existent: {PosixPath('--logtostderr --listen_host=172.28.0.12 --target_host=172.28.0.12 --tunnel_background_save_url=https'), PosixPath('//colab.research.google.com/tun/m/cc48301118ce562b961b3c22d803539adc1e0c19/gpu-t4-s-kkb-usw4a0-383nvidkrit22 --tunnel_background_save_delay=10s --tunnel_periodic_background_save_frequency=30m0s --enable_output_coalescing=true --output_coalescing_required=true ')

RuntimeError: 
        CUDA Setup failed despite GPU being available. Please run the following command to get more information:

        python -m bitsandbytes

        Inspect the output of the command and see if you can locate CUDA libraries. You might need to add them
        to your LD_LIBRARY_PATH. If you suspect a bug, please take the information from python -m bitsandbytes
        and open an issue at: https://github.com/TimDettmers/bitsandbytes/issues

## 3️⃣ Muat Model & Konfigurasi QLoRA

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Konfigurasi 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # ← ganti bfloat16 → float16
    bnb_4bit_use_double_quant=True,
)

print(f"⏳ Memuat tokenizer dari {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"⏳ Memuat model (quantized 4-bit) ...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # tidak pakai quantization
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False



print(f"\n✅ Model berhasil dimuat!")
total_params = sum(p.numel() for p in model.parameters())
print(f"   Total parameter: {total_params/1e9:.2f}B")


⏳ Memuat tokenizer dari Qwen/Qwen2.5-1.5B-Instruct ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


⏳ Memuat model (quantized 4-bit) ...


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Model berhasil dimuat!
   Total parameter: 1.54B


In [ ]:
# Konfigurasi LoRA
# ✏️ Bisa disesuaikan: r lebih besar = lebih ekspresif tapi lebih berat
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                         # LoRA rank
    lora_alpha=32,                # Scaling factor
    lora_dropout=0.05,
    bias="none",
    target_modules=[              # Layer yang di-fine-tune
        "q_proj", "k_proj", "v_proj",
        "o_proj", "gate_proj", "up_proj", "down_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 4️⃣ Training

> ✏️ **Kustomisasi training**: Sesuaikan `num_train_epochs`, `per_device_train_batch_size`, dan `learning_rate` dengan kebutuhan.  
> Untuk dataset kecil (<100 contoh), 3-5 epoch sudah cukup.


In [ ]:
OUTPUT_DIR = "./cs_model_output"

# ============================================================
# ✏️  KUSTOMISASI: Parameter training
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,    # effective batch = 2x4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    max_seq_length=512,
    packing=False,
)

def format_messages(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
)

print("🚀 Memulai training...")
trainer.train()
print("\n✅ Training selesai!")


Map:   0%|          | 0/22 [00:00<?, ? examples/s]

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

🚀 Memulai training...


Epoch,Training Loss,Validation Loss
0,No log,2.043666
1,No log,1.569858
2,9.247700,1.256911
3,9.247700,1.155809
4,5.089700,1.136014



✅ Training selesai!


## 5️⃣ Uji Model Hasil Fine-Tuning

In [ ]:
from transformers import pipeline

# Muat model terbaik hasil training
model.eval()

def chat_cs(pertanyaan, max_new_tokens=256, temperature=0.7):
    """Kirim pertanyaan ke model customer service."""
    messages = [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": pertanyaan},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()

# ============================================================
# ✏️  Ganti pertanyaan di bawah untuk uji model Anda
# ============================================================
test_questions = [
    "Halo, ada promo apa sekarang?",
    "Barang saya belum datang 5 hari, gimana ini?",
    "Cara retur produk yang rusak bagaimana ya?",
    "Bisa bayar pakai GoPay?",
]

print("=" * 60)
print("🧪 UJI MODEL CUSTOMER SERVICE")
print("=" * 60)
for q in test_questions:
    print(f"\n👤 Customer : {q}")
    print(f"🤖 AI CS    : {chat_cs(q)}")
    print("-" * 60)


🧪 UJI MODEL CUSTOMER SERVICE

👤 Customer : Halo, ada promo apa sekarang?
🤖 AI CS    : Halo! Selamat datang di TokoBagus! Saat ini kita sedang menyiapkan beberapa promosi spesial untuk kupon diskon hingga 50% pada hari Sabtu-Senin. Silakan hubungi kami melalui chat atau kontak resmi jika ingin mendaftar kupon.

Bisa saya bantu cari produk mana ya?
------------------------------------------------------------

👤 Customer : Barang saya belum datang 5 hari, gimana ini?
🤖 AI CS    : Mohon maaf atas ketidaknyamanannya! Silakan hubungi kami melalui nomor WhatsApp di [nomor] untuk informasi lebih lanjut.
------------------------------------------------------------

👤 Customer : Cara retur produk yang rusak bagaimana ya?
🤖 AI CS    : Untuk proses retur produk yang rusak, berikut langkah-langkah yang bisa anda lakukan:

1. Hubungi layanan pelanggan kami: 0800-5234 atau chat langsung di website

2. Kirim foto produk asli dan foto rusaknya

3. Tambahkan deskripsi detail tentang kerusakan (misalnya 

### 💬 Mode Interaktif — Chat Langsung dengan Model

In [ ]:
# Chat interaktif (jalankan manual)
print("💬 Mode chat interaktif. Ketik 'keluar' untuk berhenti.\n")
while True:
    try:
        user_input = input("👤 Anda: ")
        if user_input.lower() in ["keluar", "exit", "quit"]:
            print("👋 Sesi chat selesai.")
            break
        if not user_input.strip():
            continue
        response = chat_cs(user_input)
        print(f"🤖 CS   : {response}\n")
    except KeyboardInterrupt:
        print("\n👋 Sesi chat dihentikan.")
        break


## 6️⃣ Simpan Model

Pilihan:
- **Opsi A** — Simpan adapter LoRA saja (kecil, ~50 MB)  
- **Opsi B** — Merge ke model penuh lalu simpan (besar, cocok untuk deployment)
- **Opsi C** — Upload ke Hugging Face Hub


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH_DRIVE = "/content/drive/MyDrive/cs_model_qlora"

# Simpan LoRA adapter ke Google Drive
trainer.model.save_pretrained(SAVE_PATH_DRIVE)
tokenizer.save_pretrained(SAVE_PATH_DRIVE)
print(f"✅ Model adapter disimpan ke: {SAVE_PATH_DRIVE}")


Mounted at /content/drive
✅ Model adapter disimpan ke: /content/drive/MyDrive/cs_model_qlora


In [ ]:
# ============================================================
# OPSI B: Merge adapter ke model penuh (uncomment jika perlu)
# ============================================================
# from peft import PeftModel
# print("⏳ Merging model...")
# base_model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
# )
# merged_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR + "/checkpoint-best")
# merged_model = merged_model.merge_and_unload()
# MERGED_PATH = "/content/drive/MyDrive/cs_model_merged"
# merged_model.save_pretrained(MERGED_PATH)
# tokenizer.save_pretrained(MERGED_PATH)
# print(f"✅ Model merged disimpan ke: {MERGED_PATH}")
print("Sel ini opsional. Uncomment jika ingin merge ke model penuh.")


In [ ]:
# ============================================================
# OPSI C: Upload ke Hugging Face Hub (uncomment jika perlu)
# ============================================================
# from huggingface_hub import login
# login()  # Masukkan HF token Anda
# trainer.model.push_to_hub("username/nama-model-cs-indonesia")
# tokenizer.push_to_hub("username/nama-model-cs-indonesia")
# print("✅ Model berhasil diupload ke Hugging Face!")
print("Sel ini opsional. Uncomment jika ingin upload ke Hugging Face Hub.")


## 7️⃣ Muat Ulang & Inferensi dari Adapter Tersimpan

Gunakan sel ini lain kali untuk memakai model tanpa training ulang.


In [ ]:
# Muat model dari adapter yang tersimpan
# ============================================================
# from peft import PeftModel
# from google.colab import drive
# drive.mount('/content/drive')
#
# SAVE_PATH_DRIVE = "/content/drive/MyDrive/cs_model_qlora"
#
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True, bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16
# )
# base = AutoModelForCausalLM.from_pretrained(
#     "Qwen/Qwen2.5-1.5B-Instruct",
#     quantization_config=bnb_config, device_map="auto"
# )
# model = PeftModel.from_pretrained(base, SAVE_PATH_DRIVE)
# tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH_DRIVE)
# print("✅ Model berhasil dimuat dari Drive!")
print("Uncomment sel ini untuk memuat model yang sudah disimpan.")


---
## 📚 Tips & Catatan

| Topik | Saran |
|-------|-------|
| **Jumlah data** | Minimal 50 contoh per kategori untuk hasil yang baik |
| **Kualitas > kuantitas** | 100 contoh berkualitas > 1000 contoh asal-asalan |
| **Learning rate** | Mulai dari `2e-4`, turunkan jika training tidak stabil |
| **Epoch** | 3-5 untuk dataset kecil, 1-2 untuk dataset besar |
| **Evaluasi** | Selalu uji dengan kasus yang belum ada di training |
| **VRAM** | T4 (16GB) cukup. Jika OOM, kurangi `max_seq_length` |

### 🗂️ Cara Menambah Data Training
Edit variabel `raw_data` di Section 2. Setiap entri adalah dict:
```python
{"user": "pertanyaan pelanggan", "assistant": "jawaban ideal dari CS"}
```

### 🔁 Cara Menggunakan untuk Bisnis Berbeda
Ganti `SYSTEM_PROMPT` di Section 2 sesuai profil bisnis Anda, misalnya:
> *"Anda adalah asisten CS dari Klinik Sehat, klinik kecantikan dan kesehatan..."*
